##### Thai Word Segmentation - Boosted CRF + Ensemble

Notebook นี้เพิ่ม 4 แนวทาง:

1. ใช้ช่องว่างใน `ws_test.txt` เป็นตัวคั่นคำ
2. เพิ่ม `lexicon features` จาก LST20 ให้ CRF
3. ทำ ensemble กับ `lexicon_viterbi`, `newmm`, `longest`, และ `AttaCut`
4. เขียน `submission.csv` ด้วย `csv` โดยไม่ใช้ `pandas`


## 1) Install Packages

ติดตั้งแพ็กเกจหลักก่อน และให้ `AttaCut` เป็นตัวเลือกเสริมเพราะบาง environment อาจทำ dependency เพี้ยนได้

ถ้าไม่ได้ติดตั้ง `AttaCut` โค้ดจะข้าม tokenizer ตัวนี้ให้อัตโนมัติ


In [ ]:
%pip install -q python-crfsuite sklearn-crfsuite pythainlp
# %pip install -q attacut


Note: you may need to restart the kernel to use updated packages.


## 2) Configure File Paths


In [ ]:
DATA_DIR = "/kaggle/input/competitions/super-ai-engineer-ss-6-word-segmentation"
DATASET_DIR = "/kaggle/input/datasets/guntinunsawatvong/lst20-corpus-guntinun"
OUT_PATH = "submission_boosted.csv"
PRESERVE_FIRST_ROWS = 3


## 3) Imports + Helper Functions

Cell นี้รวมฟังก์ชันหลักสำหรับ parse LST20, สร้าง feature, เทรน CRF, ทำ ensemble และ export submission


In [ ]:
"""
Boosted Thai word-segmentation pipeline for the Super AI SS6 challenge.

Key ideas over the original notebook:
1. Keep whitespace in `ws_test.txt` as known hard boundaries, then score only
   non-whitespace characters. The old notebook removed spaces first and let the
   CRF predict across those gaps.
2. Add lexicon features from LST20 to the CRF.
3. Optionally ensemble the CRF with AttaCut / PyThaiNLP tokenizers.
4. Export CSV with the standard `csv` module only, so no pandas dependency.

Suggested install cell on Kaggle:
    !pip install -q python-crfsuite sklearn-crfsuite pythainlp
    # optional: !pip install -q attacut
"""

from __future__ import annotations

import argparse
import csv
import math
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import sklearn_crfsuite
from tqdm.auto import tqdm

try:
    from pythainlp.tokenize import word_tokenize
except Exception:
    word_tokenize = None

try:
    from attacut import tokenize as attacut_tokenize
except Exception:
    attacut_tokenize = None


SUBMISSION_LABELS = ["B_WORD", "I_WORD", "E_WORD"]


@dataclass
class Lexicon:
    freq: Counter
    words_by_len: dict[int, set[str]]
    lengths: list[int]
    max_len: int


@dataclass
class PredictionBundle:
    labels: list[str]
    boundary_scores: list[float]


def detect_path(base_dir: Path, candidates: list[str], must_exist: bool = True) -> Path | None:
    for name in candidates:
        candidate = base_dir / name
        if candidate.exists():
            return candidate
    if must_exist:
        raise FileNotFoundError(f"Cannot find any of {candidates} in {base_dir}")
    return None


def parse_lst20_conll(filepath: Path) -> list[list[str]]:
    sentences: list[list[str]] = []
    current: list[str] = []
    with filepath.open("r", encoding="utf-8") as handle:
        for raw_line in handle:
            line = raw_line.rstrip("\n")
            if not line or line.startswith("#"):
                if current:
                    sentences.append(current)
                    current = []
                continue
            parts = line.split("\t")
            word = parts[0]
            if word and word != "_":
                current.append(word)
    if current:
        sentences.append(current)
    return sentences


def load_lst20_split(lst20_dir: Path, split_name: str) -> list[list[str]]:
    split_dir = lst20_dir / split_name
    if not split_dir.exists():
        return []

    sentences: list[list[str]] = []
    for fp in tqdm(sorted(split_dir.rglob("*.txt")), desc=f"Load {split_name}", unit="file"):
        try:
            sentences.extend(parse_lst20_conll(fp))
        except Exception as exc:
            print(f"Skip {fp}: {exc}")
    return sentences


def word_to_bmes(word: str) -> list[str]:
    length = len(word)
    if length == 1:
        return ["S"]
    if length == 2:
        return ["B", "E"]
    return ["B"] + ["M"] * (length - 2) + ["E"]


def words_to_bmes(words: list[str]) -> list[str]:
    labels: list[str] = []
    for word in words:
        labels.extend(word_to_bmes(word))
    return labels


def bmes_to_submission(labels: list[str]) -> list[str]:
    repaired = repair_bmes(labels)
    out: list[str] = []
    for label in repaired:
        if label == "B" or label == "S":
            out.append("B_WORD")
        elif label == "M":
            out.append("I_WORD")
        else:
            out.append("E_WORD")
    return out


def repair_bmes(labels: list[str]) -> list[str]:
    repaired: list[str] = []
    index = 0
    total = len(labels)

    while index < total:
        label = labels[index]

        if label == "S":
            repaired.append("S")
            index += 1
            continue

        if label not in {"B", "M", "E"}:
            repaired.append("S")
            index += 1
            continue

        start = index
        index += 1

        while index < total and labels[index] == "M":
            index += 1

        if index < total and labels[index] == "E":
            index += 1

        length = index - start
        if length <= 1:
            repaired.append("S")
        elif length == 2:
            repaired.extend(["B", "E"])
        else:
            repaired.append("B")
            repaired.extend(["M"] * (length - 2))
            repaired.append("E")

    if len(repaired) != len(labels):
        raise ValueError("repair_bmes changed sequence length")
    return repaired


def normalize_char(char: str) -> str:
    if char.isdigit():
        return "<DIGIT>"
    if "A" <= char <= "Z" or "a" <= char <= "z":
        return char.lower()
    return char


def char_type(char: str) -> str:
    code = ord(char)
    if 0x0E01 <= code <= 0x0E2E:
        return "TH_CONS"
    if 0x0E30 <= code <= 0x0E3A:
        return "TH_VOWEL"
    if 0x0E40 <= code <= 0x0E44:
        return "TH_LEAD_VOWEL"
    if 0x0E47 <= code <= 0x0E4E:
        return "TH_MARK"
    if 0x0E00 <= code <= 0x0E7F:
        return "TH_OTHER"
    if char.isdigit():
        return "DIGIT"
    if ("A" <= char <= "Z") or ("a" <= char <= "z"):
        return "LATIN"
    if char.isspace():
        return "SPACE"
    return "PUNCT"


def build_lexicon(sentences: list[list[str]], max_len: int = 12) -> Lexicon:
    freq: Counter = Counter()
    for sentence in sentences:
        for word in sentence:
            if 1 < len(word) <= max_len:
                freq[word] += 1

    words_by_len: dict[int, set[str]] = defaultdict(set)
    for word, count in freq.items():
        length = len(word)
        min_freq = 3 if length == 2 else 2
        if count >= min_freq:
            words_by_len[length].add(word)

    lengths = sorted(words_by_len, reverse=True)
    return Lexicon(freq=freq, words_by_len=dict(words_by_len), lengths=lengths, max_len=max_len)


def compute_lexicon_stats(text: str, lexicon: Lexicon) -> dict[str, list[int]]:
    total = len(text)
    start_max = [0] * total
    end_max = [0] * total
    start_count = [0] * total
    end_count = [0] * total
    cover_delta = [0] * (total + 1)

    for length in lexicon.lengths:
        if length > total:
            continue
        vocab = lexicon.words_by_len[length]
        limit = total - length + 1
        for start in range(limit):
            piece = text[start : start + length]
            if piece in vocab:
                end = start + length - 1
                if length > start_max[start]:
                    start_max[start] = length
                if length > end_max[end]:
                    end_max[end] = length
                start_count[start] += 1
                end_count[end] += 1
                cover_delta[start] += 1
                cover_delta[end + 1] -= 1

    cover = [0] * total
    running = 0
    for index in range(total):
        running += cover_delta[index]
        cover[index] = min(running, 7)

    return {
        "lex_start_max": start_max,
        "lex_end_max": end_max,
        "lex_start_count": [min(value, 5) for value in start_count],
        "lex_end_count": [min(value, 5) for value in end_count],
        "lex_cover": cover,
    }


def char2features(chars: list[str], index: int, lex_stats: dict[str, list[int]], window: int = 4) -> dict:
    char = chars[index]
    total = len(chars)
    norm_char = normalize_char(char)
    features: dict = {
        "bias": 1.0,
        "char": norm_char,
        "char_type": char_type(char),
        "is_repeat_prev": index > 0 and chars[index - 1] == char,
        "is_repeat_next": index + 1 < total and chars[index + 1] == char,
    }

    for offset in range(1, window + 1):
        left = index - offset
        right = index + offset

        if left >= 0:
            features[f"char[-{offset}]"] = normalize_char(chars[left])
            features[f"type[-{offset}]"] = char_type(chars[left])
        else:
            features[f"BOS[{offset}]"] = True

        if right < total:
            features[f"char[+{offset}]"] = normalize_char(chars[right])
            features[f"type[+{offset}]"] = char_type(chars[right])
        else:
            features[f"EOS[{offset}]"] = True

    if index >= 1:
        features["bi_left"] = normalize_char(chars[index - 1]) + norm_char
    if index + 1 < total:
        features["bi_right"] = norm_char + normalize_char(chars[index + 1])
    if index >= 1 and index + 1 < total:
        features["tri_center"] = (
            normalize_char(chars[index - 1]) + norm_char + normalize_char(chars[index + 1])
        )

    for name, values in lex_stats.items():
        features[name] = values[index]
        if index >= 1:
            features[f"{name}[-1]"] = values[index - 1]
        if index + 1 < total:
            features[f"{name}[+1]"] = values[index + 1]

    return features


def text2features(text: str, lexicon: Lexicon, window: int = 4) -> list[dict]:
    chars = list(text)
    if not chars:
        return []
    lex_stats = compute_lexicon_stats(text, lexicon)
    return [char2features(chars, index, lex_stats, window=window) for index in range(len(chars))]


def build_dataset(sentences: list[list[str]], lexicon: Lexicon, desc: str) -> tuple[list[list[dict]], list[list[str]]]:
    features: list[list[dict]] = []
    labels: list[list[str]] = []
    for words in tqdm(sentences, desc=desc, unit="sent"):
        text = "".join(words)
        features.append(text2features(text, lexicon))
        labels.append(words_to_bmes(words))
    return features, labels


def train_crf(
    features: list[list[dict]],
    labels: list[list[str]],
    c1: float = 0.05,
    c2: float = 0.05,
    max_iterations: int = 220,
) -> sklearn_crfsuite.CRF:
    crf = sklearn_crfsuite.CRF(
        algorithm="lbfgs",
        c1=c1,
        c2=c2,
        max_iterations=max_iterations,
        all_possible_transitions=True,
    )
    crf.fit(features, labels)
    return crf


def crf_predict_bundle(crf: sklearn_crfsuite.CRF, text: str, lexicon: Lexicon) -> PredictionBundle:
    features = text2features(text, lexicon)
    labels = repair_bmes(crf.predict_single(features))

    try:
        marginals = crf.predict_marginals_single(features)
        boundary_scores = [entry.get("E", 0.0) + entry.get("S", 0.0) for entry in marginals]
    except Exception:
        boundary_scores = [1.0 if label in {"E", "S"} else 0.0 for label in labels]

    return PredictionBundle(labels=labels, boundary_scores=boundary_scores)


def ensure_words_cover_text(text: str, words: list[str], name: str) -> list[str]:
    cleaned = [word for word in words if word and not word.isspace()]
    if "".join(cleaned) != text:
        raise ValueError(f"{name} tokenizer output does not align with text")
    return cleaned


def tokenizer_bundle(text: str, words: list[str], name: str) -> PredictionBundle:
    fixed_words = ensure_words_cover_text(text, words, name)
    labels = words_to_bmes(fixed_words)
    boundary_scores = [1.0 if label in {"E", "S"} else 0.0 for label in labels]
    return PredictionBundle(labels=labels, boundary_scores=boundary_scores)


def lexicon_viterbi_words(text: str, lexicon: Lexicon) -> list[str]:
    total = len(text)
    best_score = [-10**18] * (total + 1)
    best_len = [1] * (total + 1)
    best_score[total] = 0.0

    for index in range(total - 1, -1, -1):
        chosen_score = -10**18
        chosen_len = 1

        for length in lexicon.lengths:
            if index + length > total:
                continue
            piece = text[index : index + length]
            if piece in lexicon.words_by_len[length]:
                freq = lexicon.freq[piece]
                score = math.log1p(freq) + 0.25 * length + best_score[index + length]
                if score > chosen_score:
                    chosen_score = score
                    chosen_len = length

        fallback_score = -2.5 + best_score[index + 1]
        if fallback_score > chosen_score:
            chosen_score = fallback_score
            chosen_len = 1

        best_score[index] = chosen_score
        best_len[index] = chosen_len

    words: list[str] = []
    index = 0
    while index < total:
        length = best_len[index]
        words.append(text[index : index + length])
        index += length
    return words


def probe_tokenizer(name: str, predict_fn: Callable[[str], PredictionBundle]) -> Callable[[str], PredictionBundle] | None:
    try:
        predict_fn("ภาษาไทยทดสอบ123")
        return predict_fn
    except Exception as exc:
        print(f"Skip tokenizer `{name}`: {exc}")
        return None


def available_tokenizers(lexicon: Lexicon) -> dict[str, Callable[[str], PredictionBundle]]:
    tokenizers: dict[str, Callable[[str], PredictionBundle]] = {}

    lexicon_predict = lambda text: tokenizer_bundle(text, lexicon_viterbi_words(text, lexicon), "lexicon_viterbi")
    checked = probe_tokenizer("lexicon_viterbi", lexicon_predict)
    if checked is not None:
        tokenizers["lexicon_viterbi"] = checked

    if word_tokenize is not None:
        newmm_predict = lambda text: tokenizer_bundle(
            text,
            list(word_tokenize(text, engine="newmm", keep_whitespace=False)),
            "newmm",
        )
        checked = probe_tokenizer("newmm", newmm_predict)
        if checked is not None:
            tokenizers["newmm"] = checked

        longest_predict = lambda text: tokenizer_bundle(
            text,
            list(word_tokenize(text, engine="longest", keep_whitespace=False)),
            "longest",
        )
        checked = probe_tokenizer("longest", longest_predict)
        if checked is not None:
            tokenizers["longest"] = checked

    if attacut_tokenize is not None:
        attacut_predict = lambda text: tokenizer_bundle(text, list(attacut_tokenize(text)), "attacut")
        checked = probe_tokenizer("attacut", attacut_predict)
        if checked is not None:
            tokenizers["attacut"] = checked

    return tokenizers


def count_sample_rows(sample_path: Path) -> int:
    with sample_path.open("r", encoding="utf-8") as handle:
        reader = csv.reader(handle)
        next(reader)
        return sum(1 for _ in reader)


def get_test_spans(test_path: Path, sample_path: Path) -> list[str]:
    raw = test_path.read_text(encoding="utf-8")
    expected = count_sample_rows(sample_path)

    candidate_spans = [
        re.findall(r"\S+", raw, flags=re.UNICODE),
        [re.sub(r"\s+", "", raw, flags=re.UNICODE)],
    ]

    for spans in candidate_spans:
        if sum(len(span) for span in spans) == expected:
            return [span for span in spans if span]

    raise ValueError(
        f"Could not match test characters to sample rows. "
        f"Expected {expected}, got {[sum(len(span) for span in spans) for spans in candidate_spans]}"
    )


def boundary_ensemble(
    text: str,
    bundles: dict[str, PredictionBundle],
    weights: dict[str, float],
    threshold_ratio: float,
) -> list[str]:
    total = len(text)
    if total == 0:
        return []

    max_score = sum(weights.values())
    boundaries: list[bool] = []
    for index in range(total):
        if index == total - 1:
            boundaries.append(True)
            continue

        score = 0.0
        for name, bundle in bundles.items():
            score += weights[name] * bundle.boundary_scores[index]
        boundaries.append(score >= threshold_ratio * max_score)

    labels: list[str] = []
    start = 0
    for index, is_boundary in enumerate(boundaries):
        if is_boundary:
            length = index - start + 1
            if length == 1:
                labels.append("S")
            elif length == 2:
                labels.extend(["B", "E"])
            else:
                labels.append("B")
                labels.extend(["M"] * (length - 2))
                labels.append("E")
            start = index + 1

    return repair_bmes(labels)


def evaluate_submission_f1(y_true: list[str], y_pred: list[str]) -> float:
    scores: list[float] = []
    for label in SUBMISSION_LABELS:
        tp = sum(1 for truth, pred in zip(y_true, y_pred) if truth == label and pred == label)
        fp = sum(1 for truth, pred in zip(y_true, y_pred) if truth != label and pred == label)
        fn = sum(1 for truth, pred in zip(y_true, y_pred) if truth == label and pred != label)

        if tp == 0 and fp == 0 and fn == 0:
            scores.append(0.0)
            continue

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        if precision + recall == 0:
            scores.append(0.0)
        else:
            scores.append(2.0 * precision * recall / (precision + recall))

    return sum(scores) / len(scores)


def flatten_true_labels(sentences: list[list[str]]) -> list[str]:
    flat: list[str] = []
    for words in sentences:
        flat.extend(bmes_to_submission(words_to_bmes(words)))
    return flat


def collect_eval_predictions(
    sentences: list[list[str]],
    crf: sklearn_crfsuite.CRF,
    lexicon: Lexicon,
    extra_tokenizers: dict[str, Callable[[str], PredictionBundle]],
) -> tuple[dict[str, float], list[tuple[str, dict[str, PredictionBundle]]]]:
    y_true_flat = flatten_true_labels(sentences)
    per_model_pred_flat: dict[str, list[str]] = defaultdict(list)
    eval_bundles: list[tuple[str, dict[str, PredictionBundle]]] = []

    for words in tqdm(sentences, desc="Eval predictions", unit="sent"):
        text = "".join(words)
        bundles: dict[str, PredictionBundle] = {"crf": crf_predict_bundle(crf, text, lexicon)}
        for name, predict_fn in extra_tokenizers.items():
            bundles[name] = predict_fn(text)

        eval_bundles.append((text, bundles))
        for name, bundle in bundles.items():
            per_model_pred_flat[name].extend(bmes_to_submission(bundle.labels))

    base_scores = {
        name: evaluate_submission_f1(y_true_flat, pred_flat)
        for name, pred_flat in per_model_pred_flat.items()
    }
    return base_scores, eval_bundles


def tune_ensemble(
    eval_sentences: list[list[str]],
    eval_bundles: list[tuple[str, dict[str, PredictionBundle]]],
    base_scores: dict[str, float],
) -> tuple[dict[str, float], float, float]:
    y_true_flat = flatten_true_labels(eval_sentences)

    base_weights = {name: max(score, 0.01) for name, score in base_scores.items()}
    best_score = -1.0
    best_weights = dict(base_weights)
    best_threshold = 0.5

    crf_boost_values = [1.0, 1.2, 1.4, 1.6, 1.8]
    threshold_values = [0.46, 0.48, 0.50, 0.52, 0.54, 0.56, 0.58]

    for crf_boost in crf_boost_values:
        weights = dict(base_weights)
        if "crf" in weights:
            weights["crf"] *= crf_boost

        for threshold in threshold_values:
            y_pred_flat: list[str] = []
            for text, bundles in eval_bundles:
                labels = boundary_ensemble(text, bundles, weights, threshold)
                y_pred_flat.extend(bmes_to_submission(labels))
            score = evaluate_submission_f1(y_true_flat, y_pred_flat)
            if score > best_score:
                best_score = score
                best_weights = dict(weights)
                best_threshold = threshold

    return best_weights, best_threshold, best_score


def train_crf(
    features: list[list[dict]],
    labels: list[list[str]],
    c1: float = 0.05,
    c2: float = 0.05,
    max_iterations: int = 220,
    verbose: bool = True,
) -> sklearn_crfsuite.CRF:
    print(
        f"Start CRF training | sequences={len(features):,} | "
        f"max_iterations={max_iterations} | c1={c1} | c2={c2} | verbose={verbose}",
        flush=True,
    )

    crf = sklearn_crfsuite.CRF(
        algorithm="lbfgs",
        c1=c1,
        c2=c2,
        max_iterations=max_iterations,
        all_possible_transitions=True,
        verbose=verbose,
    )
    crf.fit(features, labels)

    print("CRF training finished.", flush=True)
    return crf


def train_base_and_final_models(
    train_sentences: list[list[str]],
    eval_sentences: list[list[str]],
) -> tuple[Lexicon, sklearn_crfsuite.CRF, Lexicon, sklearn_crfsuite.CRF]:
    print(f"Build lexicon from train split: {len(train_sentences):,} sentences", flush=True)
    train_lexicon = build_lexicon(train_sentences)

    print("Extract train features ...", flush=True)
    train_x, train_y = build_dataset(train_sentences, train_lexicon, desc="Train features")

    print("Training base CRF on LST20 train split ...", flush=True)
    base_crf = train_crf(
        train_x,
        train_y,
        c1=0.05,
        c2=0.05,
        max_iterations=220,
        verbose=True,
    )

    all_sentences = train_sentences + eval_sentences
    print(f"Build lexicon from train+eval: {len(all_sentences):,} sentences", flush=True)
    final_lexicon = build_lexicon(all_sentences)

    print("Extract final features ...", flush=True)
    all_x, all_y = build_dataset(all_sentences, final_lexicon, desc="Final features")

    print("Training final CRF on train + eval ...", flush=True)
    final_crf = train_crf(
        all_x,
        all_y,
        c1=0.05,
        c2=0.05,
        max_iterations=220,
        verbose=True,
    )

    return train_lexicon, base_crf, final_lexicon, final_crf


def predict_test_labels(
    spans: list[str],
    crf: sklearn_crfsuite.CRF,
    lexicon: Lexicon,
    extra_tokenizers: dict[str, Callable[[str], PredictionBundle]],
    weights: dict[str, float] | None,
    threshold: float,
) -> list[str]:
    predicted: list[str] = []
    use_ensemble = weights is not None and len(weights) >= 2

    for span in tqdm(spans, desc="Test inference", unit="span"):
        bundles: dict[str, PredictionBundle] = {"crf": crf_predict_bundle(crf, span, lexicon)}
        for name, predict_fn in extra_tokenizers.items():
            if weights is not None and name not in weights:
                continue
            bundles[name] = predict_fn(span)

        if use_ensemble:
            labels = boundary_ensemble(span, bundles, weights, threshold)
        else:
            labels = bundles["crf"].labels
        predicted.extend(bmes_to_submission(labels))

    return predicted


def write_submission(sample_path: Path, out_path: Path, predicted: list[str], preserve_first_rows: int = 3) -> None:
    with sample_path.open("r", encoding="utf-8") as src, out_path.open(
        "w", encoding="utf-8", newline=""
    ) as dst:
        reader = csv.reader(src)
        writer = csv.writer(dst)

        header = next(reader)
        writer.writerow(header)

        for row_index, row in enumerate(reader):
            if row_index < preserve_first_rows and row[1] in SUBMISSION_LABELS:
                writer.writerow(row)
                continue
            row[1] = predicted[row_index] if row_index < len(predicted) else "B_WORD"
            writer.writerow(row)


## 4) Train, Tune, Infer, and Export

Cell นี้จะ:

1. โหลด `train` และ `eval` จาก LST20
2. เทรน base CRF บน `train` เพื่อหา ensemble weight บน `eval`
3. เทรน final CRF บน `train + eval`
4. ทำนาย `ws_test.txt`
5. สร้าง `submission_boosted.csv`


In [ ]:
data_dir = Path(DATA_DIR)
dataset_dir = Path(DATASET_DIR)
out_path = Path(OUT_PATH)

test_path = detect_path(data_dir, ["ws_test.txt", "test.txt"])
sample_path = detect_path(data_dir, ["ws_sample_submission.csv", "sample_submission.csv"])

lst20_candidates = list(dataset_dir.rglob("LST20_Corpus"))
lst20_dir = lst20_candidates[0] if lst20_candidates else dataset_dir

print(f"Test path:   {test_path}")
print(f"Sample path: {sample_path}")
print(f"LST20 dir:   {lst20_dir}")

train_sentences = load_lst20_split(lst20_dir, "train")
train_sentences = train_sentences[:50000]
eval_sentences = load_lst20_split(lst20_dir, "eval")

print(f"Train sentences: {len(train_sentences)}")
print(f"Eval sentences:  {len(eval_sentences)}")

train_lexicon, base_crf, final_lexicon, final_crf = train_base_and_final_models(
    train_sentences,
    eval_sentences,
)

Test path:   /kaggle/input/competitions/super-ai-engineer-ss-6-word-segmentation/ws_test.txt
Sample path: /kaggle/input/competitions/super-ai-engineer-ss-6-word-segmentation/ws_sample_submission.csv
LST20 dir:   /kaggle/input/datasets/guntinunsawatvong/lst20-corpus-guntinun/LST20_Corpus


Load train:   0%|          | 0/3794 [00:00<?, ?file/s]

Load eval:   0%|          | 0/474 [00:00<?, ?file/s]

Train sentences: 25000
Eval sentences:  5620
Build lexicon from train split: 25,000 sentences
Extract train features ...


Train features:   0%|          | 0/25000 [00:00<?, ?sent/s]

...

Training final CRF on train + eval ...
Start CRF training | sequences=30,620 | max_iterations=220 | c1=0.05 | c2=0.05 | verbose=True


loading training data to CRFsuite: 100%|██████████| 30620/30620 [02:35<00:00, 196.31it/s]



Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 1
0....1....2....3....4....5....6....7....8....9....10
Number of features: 101292
Seconds required: 34.584

L-BFGS optimization
c1: 0.050000
c2: 0.050000
num_memories: 6
max_iterations: 220
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

Iter 1   time=11.75 loss=2943592.87 active=100473 feature_norm=1.00
Iter 2   time=6.09  loss=1732711.97 active=98362 feature_norm=1.25
Iter 3   time=5.91  loss=1638456.81 active=100226 feature_norm=1.26
Iter 4   time=6.16  loss=1530503.64 active=98796 feature_norm=1.31
Iter 5   time=5.86  loss=1422213.68 active=99462 feature_norm=1.39
Iter 6   time=6.01  loss=972356.66 active=96705 feature_norm=2.08
Iter 7   time=12.08 loss=863866.47 active=99318 feature_norm=2.49
Iter 8   time=5.94  loss=727741.39 active=99275 feature_norm=2.85
Iter 9   time=11.89 loss=656603.52 active=99418 feature_norm=3

Eval predictions:   0%|          | 0/5620 [00:00<?, ?sent/s]

ValueError: lexicon_viterbi tokenizer output does not align with text

In [ ]:
ensemble_weights = None
ensemble_threshold = 0.5

print("Skipping Evaluation and Ensemble... Using base CRF only.")

test_spans = get_test_spans(test_path, sample_path)
print(f"Scored spans: {len(test_spans)}")
print(f"Scored chars: {sum(len(span) for span in test_spans)}")

predicted = predict_test_labels(
    test_spans,
    final_crf,
    final_lexicon,
    {},
    ensemble_weights,
    ensemble_threshold,
)

expected = count_sample_rows(sample_path)
print(f"Predicted labels: {len(predicted)}")
print(f"Sample rows:      {expected}")

if len(predicted) != expected:
    raise ValueError(f"Predicted {len(predicted)} labels, but sample has {expected} rows")

write_submission(sample_path, out_path, predicted, preserve_first_rows=PRESERVE_FIRST_ROWS)
print(f"Submission created: {out_path.resolve()}")

Skipping Evaluation and Ensemble... Using base CRF only.
Scored spans: 2067
Scored chars: 35182


Test inference:   0%|          | 0/2067 [00:00<?, ?span/s]

Predicted labels: 35182
Sample rows:      35182
Submission created: /kaggle/working/submission_boosted.csv
